# 02. Feature Engineering

01에서 정한 대로 **ΔQ 로그분산 하나**에서 시작해 다른 피처를 하나씩 더해 본다. 판단은 **Batch 1 Train 셀의 정책 단위 4-fold CV**로만 하고, Valid·Batch 2·Batch 3은 이 노트북에서 쓰지 않는다.

In [1]:
import sys; sys.path.insert(0, "..")
import numpy as np, pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from src.features import load, BASE, CHARGE, EXTRA
from src.train import cv_scores, make_model
pd.set_option("display.precision", 2)
df = load()
train = df[df.role == "train"]

## 1. 데이터 분할

Batch 1의 셀은 충전 정책마다 1~3개다. 같은 정책의 셀이 학습과 검증에 나뉘면 사실상 답을 본 상태로 검증하게 된다. 그래서 **정책 단위**로 나눈다.
- 정책 평균 수명 순으로 5구간을 만들고, 구간마다 정책 1개를 Hold-out(Valid)으로 뽑는다. Valid가 수명 범위 전체에 걸치게 하기 위해서다.
- 나머지 정책을 4개 폴드로 나눈다. `random_state=42`

In [2]:
display(df.groupby(["batch", "role"]).agg(cells=("cell", "size"), policies=("policy", "nunique"),
                                         life_min=("life", "min"), life_max=("life", "max")))
display(train.groupby("cv_fold").agg(cells=("cell", "size"), policies=("policy", "nunique"),
                                     life_min=("life", "min"), life_max=("life", "max")))
valid = df[df.role == "valid"]
assert set(train.policy).isdisjoint(valid.policy)          # Train과 Valid가 정책을 공유하지 않는다
assert train.groupby("policy").cv_fold.nunique().max() == 1  # 한 정책은 한 폴드에만 있다
print("Valid 정책:", sorted(valid.policy.unique()))

cells  policies  life_min  life_max
batch   role                                       
Batch 1 train      37        18     599.0    1227.0
        valid       9         5     534.0    1074.0
Batch 2 test       39        12     392.0    1186.0
Batch 3 batch3     44         8     541.0    1935.0

,cells,policies,life_min,life_max
cv_fold,,,,
0,10,5,599.0,917.0
1,10,5,691.0,1051.0
2,9,4,703.0,1227.0
3,8,4,617.0,1054.0


Valid 정책: ['4.4C(80%)-4.4C', '4.8C(80%)-4.8C', '5.4C(60%)-3.6C', '5.4C(80%)-5.4C', '6C(40%)-3C']


**확인.** Train 37셀(정책 18개, 4폴드)과 Valid 9셀(정책 5개)로 나뉘었고, 정책이 겹치지 않는다. Valid는 534~1,074회로 Batch 1의 짧은 쪽부터 긴 쪽까지 걸쳐 있다.

## 2. 판단 기준

- 비교 기준선: 학습 폴드의 **중앙값 수명**을 그대로 예측하는 모델
- 비교 모델: Ridge. 01에서 ΔQ와 수명이 거의 직선임을 확인했다. α는 {0.01, 0.1, 1, 10} 중 CV가 가장 낮은 값을 쓴다.
- **채택 규칙: 피처를 더했을 때 CV MAPE가 0.5%p 이상 줄어야 채택한다.** 셀이 37개뿐이라 폴드 하나에 8~10셀이 들어가고, 셀 하나가 폴드 MAPE를 1%p 넘게 움직인다. 이보다 작은 차이는 우연과 구분하기 어렵다. 이득이 작으면 피처가 적은 쪽을 택한다는 설계 원칙을 수치로 고정한 것이다.

In [3]:
ALPHAS = (0.01, 0.1, 1, 10)

def ridge_cv(cols, log_target):
    """α 그리드 중 CV 평균이 가장 낮은 결과"""
    runs = [(a, cv_scores(train, cols, make_model(Ridge(alpha=a), log_target))) for a in ALPHAS]
    a, s = min(runs, key=lambda r: np.mean(r[1]))
    return {"alpha": a, "cv_mape": np.mean(s), "cv_std": np.std(s), **{f"fold{i}": v for i, v in enumerate(s)}}

baseline = cv_scores(train, BASE, make_model(DummyRegressor(strategy="median"), False))
print(f"중앙값 기준선 CV MAPE: {np.mean(baseline):.2f}%  (폴드: {np.round(baseline, 1)})")

중앙값 기준선 CV MAPE: 19.65%  (폴드: [17.6 13.7 23.  24.3])


## 3. ΔQ 로그분산 단독

설계에서 수명 분포가 오른쪽으로 치우쳐(왜도 +0.45) `log10(cycle_life)` 타깃도 비교하기로 했다.

In [4]:
a_raw, a_log = ridge_cv(BASE, False), ridge_cv(BASE, True)
display(pd.DataFrame({"cycle_life": a_raw, "log10(cycle_life)": a_log}).T)
b1 = df[df.batch == "Batch 1"]
print("Batch 1 Pearson  ΔQ vs life:", round(np.corrcoef(b1.dq_log_var, b1.life)[0, 1], 3),
      "/ ΔQ vs log10(life):", round(np.corrcoef(b1.dq_log_var, np.log10(b1.life))[0, 1], 3))

,alpha,cv_mape,cv_std,fold0,fold1,fold2,fold3
cycle_life,0.01,8.98,2.68,7.96,7.85,13.53,6.59
log10(cycle_life),10.00,10.95,0.86,10.74,9.79,12.20,11.07


Batch 1 Pearson  ΔQ vs life: -0.886 / ΔQ vs log10(life): -0.868


**결과.** ΔQ 하나만으로 CV MAPE가 기준선 19.65%에서 **8.98%**로 절반 이하가 된다. 01의 가설대로 ΔQ가 수명 정보의 대부분을 담고 있다.

타깃은 예상과 달리 **원래 수명이 log 수명보다 약 2%p 낮다**(8.98% vs 10.95%). log 타깃은 가장 강한 규제(α=10)에서야 최저가 되어, 사실상 기울기를 줄여서 버티는 모양이다. Batch 1에서 ΔQ와의 직선 관계는 원래 단위가 조금 더 강하다(Pearson −0.886 vs −0.868). 학습 배치의 치우침도 Batch 2·3보다 작다(왜도 +0.45 vs +1.6, +1.3). log 변환으로 얻을 이점이 크지 않다. → **타깃은 cycle_life 그대로** 쓴다.

## 4. 충전 조건 추가

ΔQ에 충전 전류 P90을 더해 보고, 같은 정보로 본 첫 C-rate로 바꿔서도 더해 본다.

In [5]:
rows = {"ΔQ": a_raw}
rows.update({f"ΔQ + {name}": ridge_cv(BASE + [col], False) for name, col in CHARGE.items()})
step_b = pd.DataFrame(rows).T
step_b["vs ΔQ (%p)"] = step_b.cv_mape - a_raw["cv_mape"]
step_b

,alpha,cv_mape,cv_std,fold0,fold1,fold2,fold3,vs ΔQ (%p)
ΔQ,0.01,8.98,2.68,7.96,7.85,13.53,6.59,0.00
ΔQ + 충전 전류,10.00,10.11,1.58,7.70,11.82,9.73,11.19,1.13
ΔQ + 첫 C-rate,10.00,10.11,1.58,7.71,11.82,9.73,11.19,1.13


**결과.** 충전 전류를 더하면 CV가 **8.98% → 10.11%로 1.13%p 나빠진다.** C-rate 대체안의 결과도 소수점까지 같다. Batch 1 안에서 두 피처가 같은 정보라는 01의 관찰이 CV에서도 그대로 확인된다.

실험에서 셀마다 다른 조건은 충전 정책 하나뿐이다. 그런데 왜 더하면 나빠질까? 정책 자체가 수명을 얼마나 설명하는지, 정책만으로 예측하면 어떤지 확인한다.

In [6]:
b1 = df[df.batch == "Batch 1"]
policy_mean = b1.groupby("policy").life.transform("mean")
print("Batch 1 정책별 평균 수명이 설명하는 셀 수명 분산(R²):",
      round(1 - ((b1.life - policy_mean)**2).sum() / ((b1.life - b1.life.mean())**2).sum(), 2))
spread = b1.groupby("policy").life.agg(["size", "min", "max"]).assign(diff=lambda x: x["max"] - x["min"])
display(spread[spread["size"] > 1].sort_values("diff", ascending=False).head(3))

policy_cols = {"첫 C-rate 하나": ["first_c_rate"],
               "정책 3요소 (첫 C-rate, 전환 SOC, 두 번째 C-rate)": ["first_c_rate", "switch_soc_pct", "second_c_rate"],
               "ΔQ + 정책 3요소": BASE + ["first_c_rate", "switch_soc_pct", "second_c_rate"]}
display(pd.DataFrame({"ΔQ 하나": a_raw, **{k: ridge_cv(v, False) for k, v in policy_cols.items()}}).T[["alpha", "cv_mape", "cv_std"]])

Batch 1 정책별 평균 수명이 설명하는 셀 수명 분산(R²): 0.93


,size,min,max,diff
policy,,,,
4.8C(80%)-4.8C,2,636.0,870.0,234.0
5.4C(40%)-3.6C,2,879.0,1054.0,175.0
6C(50%)-3.6C,2,709.0,876.0,167.0


,alpha,cv_mape,cv_std
ΔQ 하나,0.01,8.98,2.68
첫 C-rate 하나,0.01,14.24,5.31
"정책 3요소 (첫 C-rate, 전환 SOC, 두 번째 C-rate)",0.01,15.11,6.44
ΔQ + 정책 3요소,0.10,10.18,1.89


**결과 → 판단.**
- **원인으로서 충전 정책은 가장 중요하다.** Batch 1에서 정책별 평균 수명만으로 셀 수명 차이의 93%가 설명된다. 다만 정책당 셀이 1~3개라 이 값은 부풀려져 있다.
- **그러나 예측 피처로는 약하다.** 정책 정보만 쓰면 CV가 14~15%로, ΔQ 하나(8.98%)보다 훨씬 나쁘다. 이유는 세 가지다.
  - CV는 처음 보는 정책의 수명을 맞혀야 한다. 첫 C-rate·전환 SOC·두 번째 C-rate 조합이 수명에 미치는 영향은 직선이 아니어서 37셀로는 배울 수 없다.
  - 같은 정책이어도 셀마다 수명이 다르다(4.8C(80%)-4.8C: 636회 vs 870회). 정책은 셀이 겪은 조건만 알려줄 뿐, 각 셀이 실제로 얼마나 상했는지는 모른다.
  - 배치가 바뀌면 같은 정책도 수명이 달라진다(03에서 6C(60%)-3C가 Batch 1 744회 → Batch 2 400회).
- **충전 정책은 원인이고, ΔQ는 그 원인이 셀에 남긴 결과를 100회 시점에 잰 값이다.** 정책의 영향이 이미 ΔQ에 담겨 있으므로, 정책을 더해도 새 정보가 없고 37셀에서는 계수 추정의 잡음만 늘어난다(ΔQ + 정책 3요소 10.18%). → **충전 조건은 채택하지 않는다.**

ESS 관점에서 둘의 쓰임은 다르다. 충전 정책은 운영자가 **고르는** 설계 변수이고, ΔQ는 이미 운전 중인 셀의 상태를 **진단하는** 값이다. 이 모델의 목적은 후자다.

## 5. 온도·저항·초기 용량 변화 추가

설계에서는 충전 전류 위에 온도·저항을 더할 계획이었다. 충전 전류를 채택하지 않았으므로 **ΔQ 단독을 기준으로** 각 후보를 하나씩 더한다. 초기 용량 변화 Qd(100) − Qd(10)도 같은 방식으로 비교한다.

In [7]:
rows = {"ΔQ": a_raw}
rows.update({f"ΔQ + {name}": ridge_cv(BASE + [col], False) for name, col in EXTRA.items()})
step_c = pd.DataFrame(rows).T
step_c["vs ΔQ (%p)"] = step_c.cv_mape - a_raw["cv_mape"]
step_c["채택"] = np.where(step_c["vs ΔQ (%p)"] <= -0.5, "O", "X")
step_c.loc["ΔQ", "채택"] = "기준"
step_c

,alpha,cv_mape,cv_std,fold0,fold1,fold2,fold3,vs ΔQ (%p),채택
ΔQ,0.01,8.98,2.68,7.96,7.85,13.53,6.59,0.00,기준
ΔQ + 평균온도,0.01,9.35,2.70,9.04,7.30,13.85,7.21,0.37,X
ΔQ + 내부저항,0.01,9.68,3.42,8.05,8.66,15.45,6.56,0.70,X
ΔQ + 초기 용량 변화,0.01,8.96,2.87,8.09,7.92,13.75,6.09,-0.02,X


**결과.** 평균온도(+0.37%p)와 내부저항(+0.70%p)은 CV를 오히려 높인다. 초기 용량 변화는 −0.02%p로 사실상 차이가 없고 폴드 편차만 커진다. 0.5%p 기준을 넘는 후보가 없다.

01에서 이 세 피처는 Batch 1에서는 상관이 있어 보였다(초기 용량 변화 +0.61). 하지만 Batch 2·3에서는 사라지거나 부호가 바뀌었다. 여기에 CV에서도 이득이 없으니, 넣었을 때 얻을 것보다 배치 이동에서 잃을 위험이 크다.

## 정리

- **ΔQ 하나로 오차가 절반 이하가 됨**
  - 중앙값 기준선 19.65% → ΔQ 로그분산 하나만 넣은 Ridge 8.98%
  - 01에서 본 "곡선 모양이 수명을 가른다"는 관찰이 예측 성능으로도 확인됨
- **log 타깃은 쓰지 않음**
  - 치우친 분포라 log가 나을 거라 예상했지만 10.95%로 오히려 2%p 나쁨
  - Batch 1에서는 ΔQ와 수명이 원래 단위에서 이미 직선이고, 치우침도 크지 않았음
- **충전 정책은 수명의 원인이지만, 예측 피처로는 ΔQ가 낫다**
  - 충전 전류 추가 시 10.11%(+1.13%p). 첫 C-rate로 바꿔도 소수점까지 같음 → Batch 1 안에서 두 피처는 같은 정보
  - 정책 평균이 Batch 1 수명 차이의 93%를 설명할 만큼 핵심 원인이지만, 정책만으로 예측하면 14~15%
  - 정책은 셀이 겪은 조건, ΔQ는 그 조건이 셀에 남긴 결과. 결과를 직접 재는 ΔQ가 있으니 원인을 더해도 새 정보가 없음
- **온도·저항·초기 용량 변화도 이득 없음**
  - 9.35% / 9.68% / 8.96%로 0.5%p 기준을 넘는 개선 없음
  - 01에서 이 피처들은 배치가 바뀌면 상관이 사라지거나 뒤집혔음 → 넣었을 때 잃을 위험이 더 큼
- **최종 입력: ΔQ 로그분산 하나, 타깃: cycle_life**
  - `src/features.py`의 `FEATURES`, `src/train.py`의 `LOG_TARGET`에 반영
  - 피처가 정해졌으니 03에서는 모델 종류만 비교